# P1 Initial Reconnaissance

## tl;dr

- Source archive and extracted files were independently verified before this notebook was created.
- Train has **776,706** rows; test has **169,011** rows; the key is unique.
- Train positives are **32,126 (4.1362%)** and occur in 10-minute temporal runs.
- Random row splits are unsafe because adjacent observations and anomaly runs would cross folds.
- Test/sample/baseline keys match exactly and in the same order.
- The organizer baseline predicts **4,402 (2.6046%)** positives; sequence-aware post-processing is a major investigation path.

## Context & Methods

This is a reproducible, read-only data-quality and structure check for the P1 water-temperature QC task. It does not train a model or create a submission.

### Key Assumptions

- The immutable source folder is the authoritative input.
- All timestamps are KST (`+09:00`) as stated in the supplied README.
- `station, year, layer, time` is the intended row key.
- A 10-minute cadence is expected within an operating segment, but real sensor gaps are allowed.

## Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
DATA_DIR = REPO_ROOT / "\ub370\uc774\ud130\uc14b \uc6d0\ubcf8" / "\ub370\uc774\ud130\uc14b_P1" / "P1_qc_anomaly"
KEY = ["station", "year", "layer", "time"]

assert DATA_DIR.exists(), f"Missing source directory: {DATA_DIR}"
DATA_DIR

WindowsPath('C:/Users/cedis/PycharmProjects/PythonProject/데이터셋 원본/데이터셋_P1/P1_qc_anomaly')

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
test = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", low_memory=False)
baseline = pd.read_csv(DATA_DIR / "baseline_rule.csv", low_memory=False)
frames = {"train": train, "test": test, "sample": sample, "baseline": baseline}

pd.DataFrame({
    name: {"rows": len(df), "columns": len(df.columns), "duplicate_keys": int(df.duplicated(KEY).sum()), "exact_duplicates": int(df.duplicated().sum())}
    for name, df in frames.items()
}).T

,rows,columns,duplicate_keys,exact_duplicates
train,776706,9,0,0
test,169011,7,0,0
sample,169011,6,0,0
baseline,169011,6,0,0


## Results

In [3]:
expected_columns = {
    "train": ["station", "year", "layer", "time", "temp", "psal", "depth", "label", "anomaly_type"],
    "test": ["station", "year", "layer", "time", "temp", "psal", "depth"],
    "sample": ["station", "year", "layer", "time", "label", "anomaly_type"],
    "baseline": ["station", "year", "layer", "time", "label", "anomaly_type"],
}
for name, df in frames.items():
    assert list(df.columns) == expected_columns[name]
    assert not df[KEY].isna().any().any()
    assert not df.duplicated(KEY).any()

assert sample[KEY].equals(test[KEY])
assert baseline[KEY].equals(test[KEY])
assert set(train["label"].unique()) <= {0, 1}
assert set(baseline["label"].unique()) <= {0, 1}

pd.Series({
    "train_test_schema_valid": True,
    "sample_keys_equal_test_in_order": sample[KEY].equals(test[KEY]),
    "baseline_keys_equal_test_in_order": baseline[KEY].equals(test[KEY]),
    "train_binary_labels": set(train["label"].unique()) <= {0, 1},
    "baseline_binary_labels": set(baseline["label"].unique()) <= {0, 1},
}, name="check")

train_test_schema_valid              True
sample_keys_equal_test_in_order      True
baseline_keys_equal_test_in_order    True
train_binary_labels                  True
baseline_binary_labels               True
Name: check, dtype: bool

In [4]:
missing = pd.concat({name: df.isna().sum() for name, df in {"train":train, "test":test}.items()}, axis=1)
missing["train_rate"] = missing["train"] / len(train)
missing["test_rate"] = missing["test"] / len(test)
missing.fillna(0)

,train,test,train_rate,test_rate
station,0,0.0,0.000000,0.000000
year,0,0.0,0.000000,0.000000
layer,0,0.0,0.000000,0.000000
time,0,0.0,0.000000,0.000000
temp,0,0.0,0.000000,0.000000
psal,16725,798.0,0.021533,0.004722
depth,1130,16368.0,0.001455,0.096846
label,0,0.0,0.000000,0.000000
anomaly_type,744580,0.0,0.958638,0.000000


In [5]:
label_summary = pd.Series({
    "normal_rows": int((train.label == 0).sum()),
    "positive_rows": int((train.label == 1).sum()),
    "positive_rate": float(train.label.mean()),
    "positive_missing_anomaly_type": int(((train.label == 1) & train.anomaly_type.isna()).sum()),
    "normal_with_anomaly_type": int(((train.label == 0) & train.anomaly_type.notna()).sum()),
})
label_summary

normal_rows                      744580.000000
positive_rows                     32126.000000
positive_rate                         0.041362
positive_missing_anomaly_type         0.000000
normal_with_anomaly_type              0.000000
dtype: float64

In [6]:
base_types = ["spike", "noise", "flatline", "offset", "drift"]
type_membership = pd.Series({
    anomaly_type: int(train.anomaly_type.fillna("").str.split("+").map(lambda parts: anomaly_type in parts).sum())
    for anomaly_type in base_types
}, name="positive_rows_with_membership")
type_membership.to_frame()

,positive_rows_with_membership
spike,104
noise,9656
flatline,6441
offset,7507
drift,8929


In [7]:
def cadence_summary(df):
    ordered = df.assign(_time=pd.to_datetime(df.time)).sort_values(["station", "year", "layer", "_time"])
    delta = ordered.groupby(["station", "year", "layer"])["_time"].diff().dt.total_seconds().div(60).dropna()
    return pd.Series({
        "timestamp_parse_failures": int(pd.to_datetime(df.time, errors="coerce").isna().sum()),
        "exact_10_min_rate": float(delta.eq(10).mean()),
        "gaps_over_10_min": int(delta.gt(10).sum()),
        "nonpositive_deltas": int(delta.le(0).sum()),
    })

pd.concat({"train": cadence_summary(train), "test": cadence_summary(test)}, axis=1)

,train,test
timestamp_parse_failures,0.000000,0.000000
exact_10_min_rate,0.998443,0.994592
gaps_over_10_min,1209.000000,914.000000
nonpositive_deltas,0.000000,0.000000


In [8]:
exception_checks = pd.Series({
    "all_time_suffixes_are_+09:00": bool(train.time.str.endswith("+09:00").all() and test.time.str.endswith("+09:00").all()),
    "gors_2026_depth_all_missing": bool(test.loc[test.station.eq("G-ORS"), "depth"].isna().all()),
    "iors_2026_layer3_rows": int((test.station.eq("I-ORS") & test.layer.eq(3)).sum()),
})
exception_checks

all_time_suffixes_are_+09:00    True
gors_2026_depth_all_missing     True
iors_2026_layer3_rows              0
dtype: object

In [9]:
baseline_summary = baseline.groupby(["station", "layer"]).agg(
    rows=("label", "size"), positives=("label", "sum"), positive_rate=("label", "mean")
)
baseline_summary.loc["ALL", "rows"] = len(baseline)
baseline_summary.loc["ALL", "positives"] = int(baseline.label.sum())
baseline_summary.loc["ALL", "positive_rate"] = float(baseline.label.mean())
baseline_summary

rows  positives  positive_rate
station layer                                    
G-ORS   1       16331.0      273.0       0.016717
I-ORS   1       16967.0      469.0       0.027642
        2        6342.0       76.0       0.011984
        4        6333.0      131.0       0.020685
        5       19456.0      636.0       0.032689
        6        5888.0      166.0       0.028193
        7       19967.0      537.0       0.026894
S-ORS   1       15208.0      490.0       0.032220
        2        5476.0      100.0       0.018262
        3        5782.0      157.0       0.027153
        4        6435.0      121.0       0.018803
        5       16795.0      683.0       0.040667
        6        6020.0      107.0       0.017774
        7        7343.0      229.0       0.031186
        8       14668.0      227.0       0.015476
ALL            169011.0     4402.0       0.026046

In [10]:
ordered = train.assign(_time=pd.to_datetime(train.time)).sort_values(["station", "year", "layer", "_time"])
grouped = ordered.groupby(["station", "year", "layer"], sort=False)
ordered["abs_temp_diff"] = grouped.temp.diff().abs()
ordered["same_as_previous_temp"] = grouped.temp.diff().eq(0)

pd.DataFrame({
    "abs_temp_diff_mean": ordered.groupby("label").abs_temp_diff.mean(),
    "abs_temp_diff_p99": ordered.groupby("label").abs_temp_diff.quantile(0.99),
    "same_as_previous_temp_rate": ordered.groupby("label").same_as_previous_temp.mean(),
})

,abs_temp_diff_mean,abs_temp_diff_p99,same_as_previous_temp_rate
label,,,
0,0.228535,2.859844,0.002012
1,1.003534,11.561425,0.194827


## Takeaways

1. Preserve chronological/group boundaries in every validation design; random row CV is leakage-prone.
2. Build type-aware detectors for spike, noise, flatline, offset, and drift, then calibrate a combined binary score.
3. Use local robust changes and sequence context instead of depending on absolute temperature, because test covers only January-June and distributions shift seasonally.
4. Respect actual observation gaps during rolling features and interval post-processing.
5. Treat G-ORS 2026 depth as structurally missing, not as an anomaly signal.
6. Do not upload a submission until the exact file passes the daily submission gate in `00_MUST_READ_FIRST.md`.